# C9-dimensionality-reduction — Review

Work through this notebook after all four lesson sessions and the practice sets.
It consolidates the complete PCA, compression, and map-reading system: derivations, NumPy idioms, fit-state contracts, degenerate eigenspaces, and honest interpretation.
Quiz answers are collapsed at the end; commit to all fifteen answers before opening them.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| PCA by centered SVD | Center $X$, then take $X_c=U\Sigma V^{\mathsf T}$ | directions are rows of $V^{\mathsf T}$; scores are $X_cV=U\Sigma$; component sample variances are $\sigma_i^2/(n-1)$ |
| Covariance eigenproblem | $C=X_c^{\mathsf T}X_c/(n-1)$ and directional variance is $u^{\mathsf T}Cu$ | stationarity on the unit sphere gives $Cu=\lambda u$; the maximum is $\lambda_{\max}$ |
| Covariance/SVD equivalence | $C=V(\Sigma^{\mathsf T}\Sigma/(n-1))V^{\mathsf T}$ | covariance eigenvalues equal $\sigma_i^2/(n-1)$ and identifiable principal subspaces agree exactly |
| Sign and repeated eigenspaces | one simple direction may flip sign; a repeated block may rotate or swap | compare a retained basis $Q$ by its projector $Q^{\mathsf T}Q$, not row-by-row |
| `NumpyPCA` fit state | fit learns a training mean, ordered row-components, variances, and full-spectrum ratios | transform reuses fitted state; sklearn/scipy PCA is not a substitute for the contract |
| Reconstruction | $Z=(X-\mu)Q^{\mathsf T}$ and $\widehat X=ZQ+\mu$ | centered reconstruction is projection by $P=Q^{\mathsf T}Q$; squared error is the dropped spectral tail |
| Truncated SVD | $W_r=U_{:,:r}\Sigma_rV^{\mathsf T}_{:r}$ is the best rank-$r$ fit | $\lVert W-W_r\rVert_F^2=\sum_{i>r}\sigma_i^2$; pair singular values and vectors and certify a budget on both sides |
| `umap-concept` | neighbor-graph layouts prioritize local neighborhoods | axes, global distances, densities, and displayed sizes have no quantitative meaning; settings and randomness matter |
| Local versus global | a view is useful only for the question its construction supports | use $k$-NN preservation locally and stretch globally; PCA projections never stretch but can create false neighbors |

## Formula and idiom sheet

**Derivation chain:**

- $\mu=n^{-1}\sum_iX_i$, $X_c=X-\mu$, and every column mean of $X_c$ is zero.
- Sample covariance: $C=X_c^{\mathsf T}X_c/(n-1)$; $C$ is symmetric positive semidefinite.
- For $\lVert u\rVert=1$, $\operatorname{Var}(X_cu)=\lVert X_cu\rVert^2/(n-1)=u^{\mathsf T}Cu$.
- A maximizing direction satisfies $Cu=\lambda u$ and has variance $\lambda$.
- From $X_c=U\Sigma V^{\mathsf T}$, covariance eigenvalues are $\sigma_i^2/(n-1)$.
- Total variance is $\operatorname{tr}(C)=\lVert X_c\rVert_F^2/(n-1)=\sum_i\lambda_i$.
- Explained ratio: $\lambda_i/\sum_j\lambda_j=\sigma_i^2/\sum_j\sigma_j^2$.
- Projector for row-basis $Q$: $P=Q^{\mathsf T}Q$, with $P^{\mathsf T}=P$ and $P^2=P$.
- Reconstruction: $\widehat X=(X-\mu)P+\mu$ and $\lVert X-\widehat X\rVert_F^2=(n-1)\sum_{j>k}\lambda_j$.

**NumPy route:**

```python
mean = X.mean(axis=0)
Xc = X - mean
C = Xc.T @ Xc / (X.shape[0] - 1)
eigenvalues, eigenvectors = np.linalg.eigh(C)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = np.maximum(eigenvalues[order], 0.0)
components = eigenvectors[:, order[:k]].T
scores = (X - mean) @ components.T
projector = components.T @ components
X_hat = scores @ components + mean
```

The independent SVD certification is:

```python
_, s, Vt = np.linalg.svd(Xc, full_matrices=False)
assert np.allclose(eigenvalues[:len(s)], s**2 / (n - 1), atol=ATOL, rtol=RTOL)
assert np.allclose(projector, Vt[:k].T @ Vt[:k], atol=ATOL, rtol=RTOL)
```

For repeated eigenvalues, the projector comparison is the contract.
For a single simple direction, `abs(v @ w)` near one is the one-dimensional special case.

**Reusable class state:**

- `mean_`: `(d,)`;
- `components_`: `(k, d)`, orthonormal rows;
- `explained_variance_`: `(k,)`, descending;
- `explained_variance_ratio_`: `(k,)`, retained eigenvalues divided by the full trace;
- `fit` returns `self`; `transform` and `inverse_transform` reject calls before fit and incompatible shapes.

Standing habits: state `ATOL` and `RTOL=0.0`; use the $n-1$ convention consistently; never compare repeated bases row-by-row; never refit inside transform; never trust a map's ruler beyond its stated guarantee.

## Self-quiz

1. Starting from the score vector $z=X_cu$, derive its sample variance as $u^{\mathsf T}Cu$.
2. Give the tangent-direction argument that forces a maximizing unit $u$ to satisfy $Cu=\lambda u$.
3. Expand $X_c^{\mathsf T}X_c/(n-1)$ through the SVD and identify every covariance eigenvalue.
4. What changes if covariance divides by $n$ instead of $n-1$: directions, eigenvalue order, reported variances, or all three?
5. A teammate's simple PC is the negative of yours.
A second teammate has a rotated pair spanning your repeated top-two eigenspace.
State the correct comparison for each.
6. Prove that $Q^{\mathsf T}Q$ is unchanged when an orthogonal matrix rotates the rows of $Q$.
7. List the four required fitted attributes of `NumpyPCA`, with shapes for $d=7$, $k=3$.
8. Why must the explained-ratio denominator include dropped eigenvalues?
9. Write the formulas for transform, inverse transform, and their combined projector form.
10. A centered matrix has rank two in five dimensions.
What are the last three covariance eigenvalues, and what reconstruction error does $k=2$ achieve?
11. What must a second call to `fit` replace, and why is recomputing the new-data mean inside `transform` wrong?
12. Write the rank-$r$ relative squared Frobenius error curve from singular values and the two-sided minimal-rank certificate.
13. Why does a truncated unit-row embedding stack generally lose the unit diagonal in $S_r=W_rW_r^{\mathsf T}$?
14. State the four `umap-concept` interpretation limits and one supported local inference.
15. State PCA projection's one-sided distance guarantee and explain why it does not prevent false neighbors.

## What to redo, per weak spot

| If you struggled with… | Redo | Reread |
|---|---|---|
| items 1–4: covariance derivation and denominator | p05, p11, p20 | Session 1 §§1–3; Session 2 §§1–3 |
| items 5–6: signs and repeated eigenspaces | p13, p21, p24 | Session 1 §4; Session 2 §4 |
| items 7–11: reusable PCA state and reconstruction | p22, p23, p24 | Session 2 §§5–8 |
| items 12–13: truncation and embedding compression | p06, p07, p09, p12, p14 | Session 3 §§2–8 |
| items 14–15: map interpretation and local/global metrics | p02, p03, p08, p10, p15–p18 | Session 4 §§3–8 |
| integration under pressure | p13, p14, p19, p23, p24 | all four sessions |

Below 12/15, follow the redo row for every miss and retake the quiz.
At 13+ you are ready to carry the fit-state and metric discipline into `C10-competition-craft`.

## Self-quiz answers

<details><summary><b>Click to reveal after answering all fifteen</b></summary>

1. $\operatorname{Var}(z)=z^{\mathsf T}z/(n-1)=u^{\mathsf T}X_c^{\mathsf T}X_cu/(n-1)=u^{\mathsf T}Cu$.
2. For every $w\perp u$, differentiate along $(u+tw)/\lVert u+tw\rVert$; stationarity gives $w^{\mathsf T}Cu=0$, so $Cu$ is parallel to $u$.
3. $C=V(\Sigma^{\mathsf T}\Sigma/(n-1))V^{\mathsf T}$; eigenvalues are $\sigma_i^2/(n-1)$ plus any required zeros.
4. Directions and order stay; every reported covariance/component variance is rescaled by $(n-1)/n$ relative to the sample convention.
5. Simple PC: compare `abs(v @ w)` or sign-fix.
Repeated pair: compare the projectors formed by the two row-bases.
6. If $R^{\mathsf T}R=I$, then $(RQ)^{\mathsf T}(RQ)=Q^{\mathsf T}R^{\mathsf T}RQ=Q^{\mathsf T}Q$.
7. `mean_ (7,)`, `components_ (3,7)`, `explained_variance_ (3,)`, `explained_variance_ratio_ (3,)`.
8. Otherwise retained ratios always sum to one and falsely claim no variance was dropped.
9. $Z=(X-\mu)Q^{\mathsf T}$; $X'=ZQ+\mu$; together $X'=(X-\mu)Q^{\mathsf T}Q+\mu$.
10. Three zeros; $k=2$ reconstructs exactly up to floating-point error.
11. All four fitted arrays are replaced.
A new mean changes the origin, so scores no longer share the training coordinate system.
12. `rel_err2 = (fro2 - np.cumsum(s**2)) / fro2`; rank $r$ is minimal when `rel_err2[r-1] <= B` and `r == 1 or rel_err2[r-2] > B`.
13. Projection drops row energy, so $(S_r)_{ii}=\lVert(W_r)_i\rVert^2$ is generally below one.
14. Local neighborhoods first; global distances/densities/sizes are not quantitative; axes have no feature meaning; settings/randomness matter.
Supported: close map neighbors are evidence of original-space neighborhood under the stated local objective.
15. Orthogonal projection never increases distance.
It can collapse a discarded-direction separation to zero, creating close-in-view pairs that were far in the original space.

</details>